# C3-gradient-descent — Review

Work through this notebook *after* the two lesson sessions and (ideally)
the practice sets.
It is a consolidation tool: a concept summary table, the update-rule and
learning-rate sheet, a 13-item self-quiz, and pointers on what to redo.
Quiz answers are collapsed at the very end — commit to your answers before
looking.

In [ ]:
import numpy as np

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Loss surfaces | Loss as a landscape over parameter space; contours = equal-loss curves | Minima where $\nabla L = \mathbf{0}$; crowded contours = steep; on the C2 MSE surface over $(w, b)$ the bottom is the best fit |
| Gradient descent | Repeat $w \leftarrow w - \eta \nabla L(w)$: steepest-descent steps that shrink near minima | On $L = (w-m)^2$ the error obeys $e_{t+1} = (1 - 2\eta)\, e_t$ — geometric convergence when the factor's size is below 1 |
| Learning rate | $\eta$ scales every step; too small crawls, too large overshoots then explodes | Divergence threshold on $a(w-m)^2$ at $\eta = 1/a$; retune by $1/n$ when a mean loss becomes a sum loss |
| Stochastic GD | Gradient estimated on a random mini-batch: noisy but cheap | $E[\text{batch grad}] = \text{full grad}$ (F5 linearity); noise floor scales with $\eta$ — shrink $\eta$ (schedule) to lower it |

## Update-rule and learning-rate sheet

**The pinned forms (C2-linear-models notation, used everywhere):**

- Model: $\hat y_i = \sum_k X_{ik} w_k + b$ (explicit bias $b$).
- Loss: $L(w, b) = \dfrac{1}{n} \sum_{i=1}^n (\hat y_i - y_i)^2$.
- Gradients: $\dfrac{\partial L}{\partial w_k} = \dfrac{2}{n} \sum_i (\hat y_i - y_i) X_{ik}$,
  $\quad \dfrac{\partial L}{\partial b} = \dfrac{2}{n} \sum_i (\hat y_i - y_i)$.
- Step: $w \leftarrow w - \eta\, \nabla_w L$, $\quad b \leftarrow b - \eta\, \dfrac{\partial L}{\partial b}$, with learning rate $\eta > 0$.

**Quadratic-bowl algebra (the exam's hand-computation core):**

- $L = a(w - m)^2$: error factor $1 - 2a\eta$ per step; closed form
  $e_t = (1 - 2a\eta)^t e_0$.
- Regimes by factor $f = 1 - 2a\eta$: monotone convergence $0 < f < 1$;
  one-step landing $f = 0$; damped oscillation $-1 < f < 0$; constant
  bounce $f = -1$; divergence $|f| > 1$ (threshold $\eta = 1/a$).
- Several dimensions: per-coordinate factors; the **steepest** curvature
  sets the $\eta$ limit. Near a non-quadratic minimum $w^\star$: local
  factor $1 - \eta L''(w^\star)$.

**Scaling traps:** sum loss = mean loss $\times\, n$ ⟹ gradients
$\times\, n$ ⟹ retune $\eta / n$; batch gradients must average (not sum)
over the batch or the scale depends on $B$; mean-MSE values are comparable
across different $n$, raw sums are not.

**Stochastic descent:** batch gradient = pinned formulas on `X[idx], y[idx]`
with $\frac1B$ scaling; unbiased ($E = $ full gradient); judge progress on
the **full-data** loss; seed every batch draw (`np.random.default_rng(SEED)`);
cost per step $\propto$ batch size; floor $\propto \eta$ ⟹ decay $\eta$
on a schedule for fast start *and* low floor.

**Code idioms (ban-register safe):**

```python
errors = (X * w).sum(axis=1) + b - y                    # (n,) residuals
grad_w = 2 / len(y) * (errors[:, None] * X).sum(axis=0)  # (d,)
grad_b = 2 / len(y) * errors.sum()
w, b = w - eta * grad_w, b - eta * grad_b                # the step (loop over steps: allowed)
```

## Self-quiz (13 items)

Answer everything before opening the collapsed answers at the very end.

**Q1.** For $L(w_1, w_2) = (w_1 + 1)^2 + 9 w_2^2$: where is the minimum,
and along which axis do the contours squeeze tighter?

**Q2.** A contour plot shows circles around the minimum; another shows long
thin ellipses. Which surface lets a single learning rate serve all
directions comfortably, and why?

**Q3.** $L(w) = w^2$, $w_0 = 2$, $\eta = 0.3$: compute $w_1$.

**Q4.** In one sentence: why does the descent step subtract
$\eta \nabla L$ rather than add it, and what F4 fact is that built on?

**Q5.** On $L(w_1, w_2) = w_1^2 + 4 w_2^2$, take one step from $(1, 1)$
with $\eta = 0.25$ and interpret the $w_2$ result.

**Q6.** For $L(w) = (w - m)^2$: state the per-step error factor and the
exact range of $\eta$ giving convergence.

**Q7.** $\eta = 0.6$ on $L(w) = (w - m)^2$: which regime (monotone /
oscillating-converging / bouncing / diverging), and why?

**Q8.** A run tuned at $\eta = 0.02$ on mean-MSE is rerun on the **sum**
loss over the same $n = 100$ points. What happens, and what $\eta$
reproduces the old run?

**Q9.** State the expected value of a mini-batch gradient (uniform random
indices) and name the two F5 tools that prove it.

**Q10.** $n = 2000$, batch size 20: per-step cost ratio of full-batch to
mini-batch descent, and the number of steps per epoch?

**Q11.** Why does constant-$\eta$ stochastic descent plateau above the
full-batch floor, and what does a decaying schedule change?

**Q12.** True or false, with the reason: "If the recorded loss ever
increases from one step to the next, the implementation must be buggy."

**Q13.** On $L(w) = (w^2 - 1)^2$, descent with small $\eta$ is started at
$w_0 = -0.5$ and at $w_0 = 2$. Where does each end up, and what general
lesson about non-quadratic surfaces is that?

## What to redo, per weak spot

- **Reading surfaces / contours (Q1, Q2, Q13):** redo p01, p06, p13(a),
  p17; reread Session 1 §§1–2.
- **Step arithmetic by hand (Q3, Q4, Q5):** redo p02, p04, p05; Session 1
  §3 and the worked example in §6.
- **Factors, regimes, thresholds (Q6, Q7):** redo p03, p07, p10, p11;
  Session 1 §§4–5.
- **Loss scaling and retuning (Q8):** redo p15; Session 1 §7 Pitfall 3.
- **Stochastic gradients (Q9, Q10, Q11):** redo p08, p12, p14, p16, p18;
  Session 2 §§3–5 and §7.
- **Run diagnosis (Q12):** redo p16 and Session 2 §7; Session 1 §7
  Pitfall 1 for the ascent signature.
- **The end-to-end fit:** if any hesitation remains, rebuild p09 from a
  blank cell — it is the unit.

## Answers (open only when done)

<details><summary><b>Answers to all 13 quiz items</b></summary>

**A1.** Minimum $(-1, 0)$. Contours squeeze along $w_2$ (coefficient 9
means steeper), so equal-loss ellipses are narrow in $w_2$.

**A2.** The circular one: equal curvature in every direction means one
$\eta$ suits all coordinates. On thin ellipses the steep direction caps
$\eta$ while the shallow direction crawls (Session 1 §4, p07).

**A3.** $L'(2) = 4$; $w_1 = 2 - 0.3 \cdot 4 = 0.8$.

**A4.** $\nabla L$ points in the direction of steepest *increase* (F4's
direction property), so descending means stepping along $-\nabla L$.

**A5.** $\nabla L(1,1) = (2, 8)$; new point $(0.5, -1)$. The $w_2$
coordinate overshot through the minimum ($1 - 8\eta = -1$: it will bounce
between $\pm 1$ forever — $\eta$ sits exactly at that coordinate's
stability boundary).

**A6.** Factor $1 - 2\eta$; convergence for exactly $0 < \eta < 1$.

**A7.** Factor $1 - 1.2 = -0.2$: oscillating-converging — alternates
sides of $m$, error shrinking $5\times$ per step.

**A8.** Sum-loss gradients are $100\times$ larger, so the old $\eta$
likely diverges (factor thrown far past $-1$ unless the bowl was very
shallow). $\eta = 0.02 / 100 = 2 \times 10^{-4}$ reproduces the run
exactly.

**A9.** $E[\text{batch gradient}] = \nabla L$, the full gradient.
Expectation of a discrete uniform variable ($\sum_i \frac1n \nabla \ell_i$)
plus linearity of expectation across the batch average (p12).

**A10.** $2000 / 20 = 100\times$ cheaper per step; $2000/20 = 100$ steps
per epoch.

**A11.** Each step adds $\eta \times$ (gradient noise) to the parameters;
near the bottom the signal vanishes but the noise does not, so the iterate
rattles in a band that scales with $\eta$. A decaying schedule shrinks the
band over time — fast early progress, low late floor (p18).

**A12.** False. Stochastic runs go uphill on individual steps while
trending down (right-on-average steps); a too-large $\eta$ also raises
loss without any coding bug. Only *full-batch* descent with a stable
$\eta$ on these losses must decrease monotonically — that special case is
a useful self-test.

**A13.** $w_0 = -0.5 \to$ the minimum at $-1$; $w_0 = 2 \to$ the minimum
at $+1$ (each start slides into its own valley). Lesson: on non-quadratic
surfaces descent finds *a nearby* minimum, not a global best — the
starting point is part of the algorithm (p17).

</details>